# Customer Churn Prediction using Data Analytics and AI

**IBM SkillsBuild Academic Internship | Big Data & Business Management / Data Analytics**

This notebook looks at customer churn in a telecom-style dataset, checks a few patterns, and compares three starter classifiers. The data currently in `data/raw` is a synthetic teaching copy with the same columns as IBM's public sample because I could not retrieve the original CSV in this workspace. It should not be described as real company data.


## 1. Imports and load data

I'm keeping the original CSV under `data/raw` and writing a cleaned copy separately so I can always go back to the first version. Run the notebook from the project root.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from modeling import prepare_customer_data, train_models, FEATURES

raw_path = ROOT / 'data' / 'raw' / 'Telco-Customer-Churn.csv'
raw_df = pd.read_csv(raw_path)
print('Rows and columns:', raw_df.shape)
raw_df.head()

## 2. A quick look at the data quality

`TotalCharges` has blanks for brand-new accounts in the source-style format. For those rows, I use zero because there has not been time to accumulate a billed total. I also check for repeated rows and convert the charge columns to numeric before plotting.

In [ ]:
print(raw_df.dtypes)
print('Duplicate rows:', raw_df.duplicated().sum())
print('Blank total charges:', raw_df['TotalCharges'].astype(str).str.strip().eq('').sum())

churn_df = prepare_customer_data(raw_df)
clean_path = ROOT / 'data' / 'cleaned' / 'telco_churn_cleaned.csv'
clean_path.parent.mkdir(parents=True, exist_ok=True)
churn_df.to_csv(clean_path, index=False)
print('Cleaned shape:', churn_df.shape)
print('Churn rate:', round((churn_df['Churn'] == 'Yes').mean() * 100, 1), '%')
churn_df[['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'service_count']].describe()

## 3. Exploratory analysis

These are group-level rates. They help me decide which questions to investigate, but they do not show that any one customer attribute causes churn.

In [ ]:
sns.set_theme(style='whitegrid', palette='deep')


### Chart 1: Contract type

**What I notice:** Month-to-month customers stand out as the group with the highest churn in this sample. Longer contracts may be a useful retention discussion, though a contract choice alone doesn't explain every departure.

In [ ]:
plot_data = churn_df.groupby('Contract', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_data, x='Contract', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by contract type', xlabel='', ylabel='Churn rate')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### Chart 2: Tenure

**What I notice:** The churn rate is higher among newer accounts. That points to the first few months as a practical time to improve onboarding and check for early service issues.

In [ ]:
plot_data = churn_df.groupby('tenure_bucket', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=plot_data, x='tenure_bucket', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by tenure group', xlabel='Tenure', ylabel='Churn rate')
plt.tight_layout()
plt.show()

### Chart 3: Monthly charges

**What I notice:** Customers who leave tend to have a somewhat higher monthly bill here. Price could be part of the story, but it may also reflect which services they use.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=churn_df, x='Churn', y='MonthlyCharges', ax=ax)
ax.set_title('Monthly charges by churn outcome')
plt.tight_layout()
plt.show()

### Chart 4: Internet service

**What I notice:** The internet service groups do not have the same churn rate. I'd look more closely at fiber customers' support experience and bills before drawing conclusions.

In [ ]:
plot_data = churn_df.groupby('InternetService', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_data, x='InternetService', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by internet service', xlabel='', ylabel='Churn rate')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### Chart 5: Payment method

**What I notice:** Electronic check users appear to leave more often in this sample. A payment method might be a signal of customer habits or friction, rather than a reason by itself.

In [ ]:
plot_data = churn_df.groupby('PaymentMethod', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_data, x='PaymentMethod', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by payment method', xlabel='', ylabel='Churn rate')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### Chart 6: Senior citizen

**What I notice:** The senior-citizen group has a different churn rate, though this binary flag is a rough demographic category and should not be used to stereotype customers.

In [ ]:
plot_data = churn_df.groupby('SeniorCitizen', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_data, x='SeniorCitizen', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by senior citizen', xlabel='', ylabel='Churn rate')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### Chart 7: Online security

**What I notice:** Customers without online security show more churn in this sample. It could be a marker for service mix, so I'd treat it as a lead for a service review.

In [ ]:
plot_data = churn_df.groupby('OnlineSecurity', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_data, x='OnlineSecurity', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by online security', xlabel='', ylabel='Churn rate')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### Chart 8: Tech support

**What I notice:** No tech support is associated with higher churn in the group comparison. Proactive help might be worth offering to customers who have unresolved issues.

In [ ]:
plot_data = churn_df.groupby('TechSupport', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False).reset_index(name='churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_data, x='TechSupport', y='churn_rate', color='#6286a5', ax=ax)
ax.set(title='Churn rate by tech support', xlabel='', ylabel='Churn rate')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 4. Feature engineering notes

I grouped tenure into four readable ranges because exact month values make the charts noisy. `service_count` counts services marked Yes, while `avg_monthly_spend` is total charges divided by tenure (and monthly charges for a brand-new account). These are small summaries that may help a model find broader patterns.

In [ ]:
churn_df[['tenure', 'tenure_bucket', 'service_count', 'TotalCharges', 'avg_monthly_spend']].head(10)

## 5. Model comparison

The preprocessing and training live in `src/modeling.py` so the dashboard can use exactly the same transformations. I use a stratified holdout set. The model is selected by ROC-AUC, while precision/recall make the false-alarm versus missed-churn tradeoff more visible.

In [ ]:
data, results, selected_model, best_name, X_test, y_test = train_models()
metrics = pd.DataFrame(results).T
metrics[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].sort_values('roc_auc', ascending=False)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, model_info) in zip(axes, results.items()):
    # The saved metrics include counts in the order: actual no/yes by predicted no/yes.
    ConfusionMatrixDisplay(np.array(model_info['confusion_matrix']), display_labels=['Stayed', 'Churned']).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()
print('Selected model:', best_name)

## 6. Business recommendations

1. Test a small retention offer for month-to-month customers before renewal, and measure whether the offer actually reduces churn.
2. Add a friendly check-in during the first three months, especially after installation or a support contact.
3. Review fiber customers' bills and support complaints together; the chart alone cannot tell whether price, service quality, or both are involved.
4. Make automatic payment setup easier, but keep mailed and electronic options available. The payment pattern may be a proxy rather than a cause.
5. Offer practical security and tech-support guidance when customers sign up, then track take-up and subsequent churn.

These are tests to run with customers, not guaranteed fixes. The dataset is synthetic, which limits how much can be claimed.

## 7. Limitations and next steps

This dataset is synthetic and uses deliberately designed patterns, so the model scores are only a workflow demonstration. In a real study, I would get permission to use a real customer dataset, check for leakage and fairness issues, tune the probability threshold based on retention-team capacity, and estimate the cost of false positives and missed churn. I also haven't tested whether the model stays accurate over time.